# Revela — Notebook 07: BCN20000 Cancer-Risk CNN Evaluation

**Issue:** #120 — D3.5: Evaluate dermoscopic cancer-risk CNN  
**Status:** Complete  
**Depends on:** #119 — Retrain dermoscopic CNN (closed)  
**Script:** `src/model/evaluate_cancer_risk.py`  
**Config:** `config/bcn20000_cancer_risk_config.yaml`  

### Purpose
Evaluate the best checkpoint from #119 on the frozen test split and report all metrics required by #120:
- Overall accuracy, macro-F1, balanced accuracy
- Class-wise precision, recall, F1
- Cancer / malignant recall and false-negative rate (primary metric)
- Melanoma recall and NMSC recall separately
- Confusion matrix (counts and row-normalized)
- Comparison vs CNN v1

### Deliverables
- `outputs/metrics/bcn20000_cancer_risk_test_metrics.json`
- `outputs/metrics/bcn20000_cancer_risk_classification_report.csv`
- `outputs/plots/bcn20000_cancer_risk_confusion_matrix.png`
- `docs/model/bcn20000_cancer_risk_evaluation_summary.md`
- `src/model/evaluate_cancer_risk.py`

### Priority metrics
1. Cancer / malignant recall (Melanoma + NMSC combined)
2. Cancer / malignant false-negative rate
3. Melanoma recall
4. Non-melanoma skin cancer recall
5. Macro-F1
6. Balanced accuracy

> Test split is frozen — built in #118, never used during training or model selection in #119.

## Block 1 — Evaluation Pipeline Overview

| Item | Detail |
|------|--------|
| Entry point | `python3 -m src.model.evaluate_cancer_risk` |
| Config | `config/bcn20000_cancer_risk_config.yaml` |
| Test CSV | `data/processed/bcn20000_cancer_risk/test.csv` (2,659 rows) |
| Checkpoint | `models/bcn20000_cancer_risk_effnet_b0/best_model.pth` (epoch 6) |
| Architecture | EfficientNet-B0, 4 output classes |
| Classes | Melanoma / Non-melanoma skin cancer / Benign nevus / Other non-cancer / indeterminate lesion |
| Device | MPS (Apple Silicon) or CPU |
| Best model criterion | Validation macro-F1 (selected in #119) |

The eval script mirrors `src/model/evaluate.py` (CNN v1) with three additions:
- `cancer_recall` and `cancer_fnr` (combined Melanoma + NMSC)
- `melanoma_recall` and `nmsc_recall` individually
- Row-normalized confusion matrix saved to JSON

## Block 2 — Test Split Verification

Confirm the test split was held out correctly and was not used during training or model selection.

In [ ]:
import pandas as pd

BASE = '../../../data/processed/bcn20000_cancer_risk'
CLASS_ORDER = [
    'Melanoma',
    'Non-melanoma skin cancer',
    'Benign nevus',
    'Other non-cancer / indeterminate lesion',
]

test  = pd.read_csv(f'{BASE}/test.csv')
train = pd.read_csv(f'{BASE}/train.csv')
val   = pd.read_csv(f'{BASE}/val.csv')

print('=== SPLIT SIZES ===')
print(f'  Train : {len(train):,} rows')
print(f'  Val   : {len(val):,} rows')
print(f'  Test  : {len(test):,} rows')
print(f'  Total : {len(train) + len(val) + len(test):,} rows')
print()

train_val_lesions = set(train['lesion_id']) | set(val['lesion_id'])
test_lesions = set(test['lesion_id'])
overlap = train_val_lesions & test_lesions
print(f'Lesion-level leakage check: {len(overlap)} shared lesion_ids between test and train/val')
print('PASS — no leakage' if len(overlap) == 0 else 'FAIL — leakage detected')
print()

print('=== TEST SET CLASS DISTRIBUTION ===')
print(f'{"Class":<48} {"Count":>5}  {"   %":>5}')
print('-' * 62)
for cls in CLASS_ORDER:
    n = (test['class_label'] == cls).sum()
    print(f'{cls:<48} {n:>5}  {n/len(test)*100:>5.1f}%')
print('-' * 62)
print(f'{"TOTAL":<48} {len(test):>5} {100.0:>5.1f}%')

=== SPLIT SIZES ===
  Train : 12,352 rows
  Val   : 2,628 rows
  Test  : 2,659 rows
  Total : 17,639 rows

Lesion-level leakage check: 0 shared lesion_ids between test and train/val
PASS — no leakage

=== TEST SET CLASS DISTRIBUTION ===
Class                                            Count      %
--------------------------------------------------------------
Melanoma                                           572   21.5%
Non-melanoma skin cancer                           656   24.7%
Benign nevus                                       935   35.2%
Other non-cancer / indeterminate lesion            496   18.7%
--------------------------------------------------------------
TOTAL                                             2659 100.0%


## Block 3 — Smoke Test

Run the eval script on 2 batches to verify the pipeline loads the checkpoint, runs inference, and writes outputs.

```bash
python3 -m src.model.evaluate_cancer_risk \
  --config config/bcn20000_cancer_risk_config.yaml \
  --batch-size 4 --num-workers 0 --max-batches 2
```

In [ ]:
import subprocess

result = subprocess.run(
    [
        'python3', '-m', 'src.model.evaluate_cancer_risk',
        '--config', 'config/bcn20000_cancer_risk_config.yaml',
        '--batch-size', '4',
        '--num-workers', '0',
        '--max-batches', '2',
    ],
    capture_output=True,
    text=True,
    cwd='../../..',
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
else:
    print('SMOKE TEST PASSED — pipeline runs end-to-end (8 samples, 2 batches × 4)')

Evaluation results:
  - test_examples:     8
  - top1_accuracy:     0.3750
  - macro_f1:          0.2833
  - balanced_accuracy: 0.2167
  - cancer_recall:     0.6667  (Melanoma + NMSC predicted as any malignant class)
  - cancer_fnr:        0.3333
  - melanoma_recall:   0.0000
  - nmsc_recall:       0.6667

Class-wise metrics:
  - Melanoma: precision=0.0000, recall=0.0000, f1=0.0000, support=0
  - Non-melanoma skin cancer: precision=1.0000, recall=0.6667, f1=0.8000, support=3
  - Benign nevus: precision=1.0000, recall=0.2000, f1=0.3333, support=5
  - Other non-cancer / indeterminate lesion: precision=0.0000, recall=0.0000, f1=0.0000, support=0

Saved metrics JSON:           outputs/metrics/bcn20000_cancer_risk_test_metrics.json
Saved classification report:  outputs/metrics/bcn20000_cancer_risk_classification_report.csv
Saved confusion matrix PNG:   outputs/plots/bcn20000_cancer_risk_confusion_matrix.png

SMOKE TEST PASSED — pipeline runs end-to-end (8 samples, 2 batches × 4)


## Block 4 — Full Evaluation Run

Evaluate on all 2,659 test rows.

```bash
python3 -m src.model.evaluate_cancer_risk \
  --config config/bcn20000_cancer_risk_config.yaml
```

In [ ]:
import subprocess

result = subprocess.run(
    [
        'python3', '-m', 'src.model.evaluate_cancer_risk',
        '--config', 'config/bcn20000_cancer_risk_config.yaml',
    ],
    capture_output=True,
    text=True,
    cwd='../../..',
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

Evaluation results:
  - test_examples:     2659
  - top1_accuracy:     0.6777
  - macro_f1:          0.6581
  - balanced_accuracy: 0.6585
  - cancer_recall:     0.7191  (Melanoma + NMSC predicted as any malignant class)
  - cancer_fnr:        0.2809
  - melanoma_recall:   0.5787
  - nmsc_recall:       0.7241

Class-wise metrics:
  - Melanoma: precision=0.6051, recall=0.5787, f1=0.5916, support=572
  - Non-melanoma skin cancer: precision=0.7285, recall=0.7241, f1=0.7263, support=656
  - Benign nevus: precision=0.7680, recall=0.7647, f1=0.7663, support=935
  - Other non-cancer / indeterminate lesion: precision=0.5312, recall=0.5665, f1=0.5483, support=496

Confusion matrix:
true\pred           Melanoma            Non-melanoma skin   Benign nevus        Other non-cancer /
Melanoma            331                 37                  132                 72
Non-melanoma skin   40                  475                 26                  115
Benign nevus        126                 33           

## Block 5 — Headline Metrics

In [ ]:
import json

with open('../../../outputs/metrics/bcn20000_cancer_risk_test_metrics.json') as f:
    metrics = json.load(f)

print('=== HEADLINE METRICS ===')
print(f'  Checkpoint epoch         : {metrics["checkpoint_epoch"]}')
print(f'  Test examples            : {metrics["num_test_examples"]:,}')
print()
print(f'  Top-1 accuracy           : {metrics["top1_accuracy"]:.4f}  ({metrics["top1_accuracy"]*100:.2f}%)')
print(f'  Macro-F1                 : {metrics["macro_f1"]:.4f}')
print(f'  Balanced accuracy        : {metrics["balanced_accuracy"]:.4f}  ({metrics["balanced_accuracy"]*100:.2f}%)')
print()
print(f'  Cancer / malignant recall: {metrics["cancer_recall"]:.4f}  ({metrics["cancer_recall"]*100:.2f}%)')
print(f'  Cancer / malignant FNR   : {metrics["cancer_fnr"]:.4f}  ({metrics["cancer_fnr"]*100:.2f}%)')
print(f'  Melanoma recall          : {metrics["melanoma_recall"]:.4f}  ({metrics["melanoma_recall"]*100:.2f}%)')
print(f'  NMSC recall              : {metrics["nmsc_recall"]:.4f}  ({metrics["nmsc_recall"]*100:.2f}%)')

=== HEADLINE METRICS ===
  Checkpoint epoch         : 6
  Test examples            : 2,659

  Top-1 accuracy           : 0.6777  (67.77%)
  Macro-F1                 : 0.6581
  Balanced accuracy        : 0.6585  (65.85%)

  Cancer / malignant recall: 0.7191  (71.91%)
  Cancer / malignant FNR   : 0.2809  (28.09%)
  Melanoma recall          : 0.5787  (57.87%)
  NMSC recall              : 0.7241  (72.41%)


### Results Summary

| Metric | Value |
|--------|-------|
| **Cancer / malignant recall** | **71.91%** |
| **Cancer / malignant FNR** | **28.09%** |
| Melanoma recall | 57.87% |
| NMSC recall | 72.41% |
| Macro-F1 | 0.6581 |
| Balanced accuracy | 65.85% |
| Top-1 accuracy | 67.77% |

Cancer recall counts any true malignant case (Melanoma or NMSC) predicted as *any* malignant class (Melanoma or NMSC).

## Block 6 — Class-Wise Metrics

In [ ]:
import pandas as pd

report = pd.read_csv('../../../outputs/metrics/bcn20000_cancer_risk_classification_report.csv')

print('=== CLASS-WISE METRICS ===')
print(f'{"Class":<48} {"Precision":>10} {"Recall":>8} {"F1":>8} {"Support":>8}')
print('-' * 86)
for _, row in report.iterrows():
    print(
        f'{row["class_name"]:<48}'
        f'{float(row["precision"])*100:>9.2f}%'
        f'{float(row["recall"])*100:>7.2f}%'
        f'{float(row["f1"])*100:>7.2f}%'
        f'{int(row["support"]):>8}'
    )

=== CLASS-WISE METRICS ===
Class                                            Precision   Recall       F1  Support
--------------------------------------------------------------------------------------
Melanoma                                             60.51%   57.87%   59.16%      572
Non-melanoma skin cancer                             72.85%   72.41%   72.63%      656
Benign nevus                                         76.80%   76.47%   76.63%      935
Other non-cancer / indeterminate lesion              53.12%   56.65%   54.83%      496


## Block 7 — Confusion Matrix

In [ ]:
import json

with open('../../../outputs/metrics/bcn20000_cancer_risk_test_metrics.json') as f:
    metrics = json.load(f)

cm      = metrics['confusion_matrix']
cm_norm = metrics['confusion_matrix_normalized']
short   = ['Melanoma', 'NMSC', 'Benign nevus', 'Other']

print('=== CONFUSION MATRIX (counts) ===')
print(f'{"":>14}' + ''.join(f'{s:>14}' for s in short))
for i, row in enumerate(cm):
    print(f'True {short[i]:<9}' + ''.join(f'{v:>14}' for v in row))
print()

print('=== CONFUSION MATRIX (row-normalized %) ===')
print(f'{"":>14}' + ''.join(f'{s:>14}' for s in short))
for i, row in enumerate(cm_norm):
    print(f'True {short[i]:<9}' + ''.join(f'{v*100:>13.1f}%' for v in row))

=== CONFUSION MATRIX (counts) ===
                  Melanoma          NMSC  Benign nevus         Other
True Melanoma          331            37           132            72
True NMSC               40           475            26           115
True Benign nevus      126            33           715            61
True Other              50           107            58           281

=== CONFUSION MATRIX (row-normalized %) ===
                  Melanoma          NMSC  Benign nevus         Other
True Melanoma        57.9%          6.5%         23.1%         12.6%
True NMSC             6.1%         72.4%          4.0%         17.5%
True Benign nevus    13.5%          3.5%         76.5%          6.5%
True Other           10.1%         21.6%         11.7%         56.7%


**Key misclassification patterns:**
- 132 Melanoma cases (23.1%) predicted as Benign nevus — most critical error
- 115 NMSC cases (17.5%) predicted as Other — second largest error
- 107 Other cases (21.6%) predicted as NMSC — inflates NMSC false positives

Confusion matrix PNG: `outputs/plots/bcn20000_cancer_risk_confusion_matrix.png`

## Block 8 — Comparison vs CNN v1

CNN v1 used a 3-class taxonomy (Melanoma / Benign nevus / Other lesion). CNN v1's "Other lesion" silently contained all NMSC cases.  
Both models evaluated on 2,659-row BCN20000 test subsets, seed=42, 15% hold-out (same image subset).

**Label mapping caveat:** class-by-class recall comparison is not meaningful. Only aggregate metrics and melanoma recall can be compared fairly. CNN v1 cancer recall cannot be derived from cached metrics — NMSC was inside "Other lesion".

In [ ]:
import json

with open('../../../outputs/metrics/bcn20000_cancer_risk_test_metrics.json') as f:
    v2 = json.load(f)

# CNN v1 cached metrics from docs/decision_log.md DEC-007
v1 = {
    'top1_accuracy': 0.7616,
    'macro_f1': 0.7443,
    'balanced_accuracy': 0.7529,
    'melanoma_recall': 0.6958,
    'nmsc_recall': None,
    'cancer_recall': None,
    'cancer_fnr': None,
}

def fmt(v, as_pct=True):
    if v is None:
        return '— (not computable)'
    return f'{v*100:.2f}%' if as_pct else f'{v:.4f}'

def delta(v2_val, v1_val, as_pct=True):
    if v1_val is None:
        return 'now measurable'
    diff = v2_val - v1_val
    sign = '+' if diff >= 0 else ''
    return f'{sign}{diff*100:.1f} pp' if as_pct else f'{sign}{diff:.4f}'

rows = [
    ('Top-1 accuracy',    v2['top1_accuracy'],    v1['top1_accuracy'],    True),
    ('Macro-F1',          v2['macro_f1'],          v1['macro_f1'],         False),
    ('Balanced accuracy', v2['balanced_accuracy'], v1['balanced_accuracy'], True),
    ('Melanoma recall',   v2['melanoma_recall'],   v1['melanoma_recall'],  True),
    ('NMSC recall',       v2['nmsc_recall'],        v1['nmsc_recall'],      True),
    ('Cancer recall',     v2['cancer_recall'],      v1['cancer_recall'],    True),
    ('Cancer FNR',        v2['cancer_fnr'],         v1['cancer_fnr'],       True),
]

print('=== CNN v1 (3-class) vs CNN v2 (4-class cancer-risk) ===')
print()
print(f'{"Metric":<28} {"CNN v1 (3-class)":>20} {"CNN v2 (4-class)":>18} {"Change":>16}')
print('-' * 82)
for label, v2_val, v1_val, as_pct in rows:
    print(f'{label:<28} {fmt(v1_val, as_pct):>20} {fmt(v2_val, as_pct):>18} {delta(v2_val, v1_val, as_pct):>16}')

=== CNN v1 (3-class) vs CNN v2 (4-class cancer-risk) ===

Metric                        CNN v1 (3-class)   CNN v2 (4-class)           Change
----------------------------------------------------------------------------------
Top-1 accuracy                          76.16%             67.77%         -8.4 pp
Macro-F1                                0.7443             0.6581          -0.0862
Balanced accuracy                       75.29%             65.85%         -9.4 pp
Melanoma recall                         69.58%             57.87%        -11.7 pp
NMSC recall              — (not computable)             72.41%    now measurable
Cancer recall            — (not computable)             71.91%    now measurable
Cancer FNR               — (not computable)             28.09%    now measurable


## Block 9 — Does the New Taxonomy Better Fit the Product Objective?

**Yes — for the product objective, not for raw scores.**

Revela is a **clinical reasoning scaffold for dermatology residents**, not a binary safe/not-safe device. Under CNN v1, a resident seeing "Other lesion" for a BCC or SCC image received zero signal about cancer risk — that class conflated melanocytic nevi, actinic keratoses, and thousands of NMSC cases under one uninformative label. CNN v2 surfaces NMSC explicitly (72.4% recall) and makes cancer recall a first-class metric (71.9%), which directly teaches the melanoma-vs-NMSC distinction residents need to learn.

The drop in aggregate accuracy (−8.4 pp) reflects a **genuinely harder task**, not a failure of the taxonomy. Separating four clinically meaningful categories is harder than three. The confusion matrix shows the model makes **clinically interpretable errors** — Melanoma confused with Benign nevus (not NMSC), NMSC confused with Other (not Benign nevus) — rather than random noise. For an educational prototype, a model that is wrong in instructive ways and surfaces the right risk categories is more aligned with the product goal than a higher-accuracy model that cannot distinguish NMSC from benign conditions.

### Communication rules

> This is an **educational prototype**, not a clinical diagnostic device.  
> Do not interpret cancer recall as "the model detects X% of all skin cancers."  
> Do not claim clinical readiness or use these results to support diagnosis decisions.

## Block 10 — Known Limitations

1. **Skin-tone (FST) distribution unknown.** BCN20000 is from a single institution (Hospital Clínic de Barcelona). Performance on darker skin tones may differ.
2. **Class imbalance.** Benign nevus (935 rows) is the largest test class; Other (496 rows) is the smallest. Read the confusion matrix with support counts in mind.
3. **Out-of-distribution generalization.** All images are from one institution's dermoscopy archive. Performance on images from different devices, settings, or patient populations is unknown.
4. **Actinic keratosis (AK).** AK is merged into "Other non-cancer / indeterminate lesion" per DEC-008. The model does not distinguish AK from other indeterminate lesions.
5. **No calibration.** Softmax confidence scores are not calibrated — do not treat them as true posterior probabilities.

## Block 11 — Summary

| Item | Detail |
|------|--------|
| Issue | #120 — D3.5 |
| Model | EfficientNet-B0, epoch 6 checkpoint |
| Test split | 2,659 rows, frozen, seed=42, 0 lesion-level leaks |
| **Cancer recall** | **71.91%** |
| Cancer FNR | 28.09% |
| Melanoma recall | 57.87% |
| NMSC recall | 72.41% |
| Macro-F1 | 0.6581 |
| Balanced accuracy | 65.85% |
| Top-1 accuracy | 67.77% |

### Deliverable checklist

- [x] Test split verified — leakage-free (Block 2)
- [x] Smoke test passes (Block 3)
- [x] Full evaluation complete — 2,659 rows (Block 4)
- [x] `outputs/metrics/bcn20000_cancer_risk_test_metrics.json` saved
- [x] `outputs/metrics/bcn20000_cancer_risk_classification_report.csv` saved
- [x] `outputs/plots/bcn20000_cancer_risk_confusion_matrix.png` saved
- [x] `docs/model/bcn20000_cancer_risk_evaluation_summary.md` written
- [x] CNN v1 comparison complete (Block 8)
- [x] GitHub issue #120 updated

### Dependency chain
```
D3.1 — Approve cancer-risk taxonomy    ✅ (#116)
  → D3.2 (#117) — Create label mapping  ✅
    → D3.3 (#118) — Rebuild splits       ✅
      → D3.4 (#119) — Retrain CNN         ✅
        → D3.5 (#120) — Evaluate CNN       ✅ THIS NOTEBOOK
          → D3.6 (#121) — Assess supplemental datasets  ← NEXT
```